# Explanations: what the app will say about each lounge and area

Each decision (`../scorecard.ipynb`, `../growth.ipynb`) gets a pyramid: a one-sentence answer, then the arguments that
matter most for it (ranked by `src.explain.prioritize`, never by the writer), each backed by 2-5
numbers copied from the decision's fact sheet. `src.explain.verify` rejects any explanation that
cites a value or writes a number not on the fact sheet.

The **AI** explanations here were written in a Claude Code session from the same prompts
`just explain` sends to the API (`python -m src.explain prompts|check|ingest`), so no API credits
were spent; they are cached in `data/explanations/cache.json` by a hash of their fact sheet, so a
changed number falls back to the deterministic **template**. The NOT SCORED airport lounge and
SKIP areas always use the template.

Data: `src.explain.v3_subjects()` (baseline, medium levels).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from collections import Counter

import pandas as pd
from IPython.display import Markdown, display

from src.explain import (LOUNGE_TABLE, AREA_TABLE, explain, load_cache, table_rows,
                         v3_subjects, verify)

pd.set_option("display.max_colwidth", 120)
cache = load_cache()
subjects = v3_subjects()
exps = {(k, sid): explain(k, sid, a, f, cache=cache) for k, sid, a, f in subjects}
facts = {(k, sid): f for k, sid, a, f in subjects}
print(Counter((k, e.source) for (k, _), e in exps.items()))
print("grounding failures:", sum(bool(verify(e, facts[key], key[0])) for key, e in exps.items()))

Counter({('area', 'template'): 464, ('area', 'ai'): 97, ('lounge', 'ai'): 23, ('uae', 'ai'): 1, ('lounge', 'template'): 1})
grounding failures: 0


In [2]:
def show(kind, sid):
    e, f = exps[(kind, sid)], facts[(kind, sid)]
    lines = [f"### {sid} — **{e.action}** ({e.source})", "", f"**{e.headline}**", ""]
    for r in e.reasons:
        ev = "; ".join(f"{x.label}: {x.value}" for x in r.evidence)
        lines += [f"- **{r.topic}**: {r.claim}  \n  <small>{ev}</small>"]
    lines += ["", f"_{e.table_caption}_"]
    display(Markdown("\n".join(lines)))


def table(kind, sid):
    fields = {"lounge": LOUNGE_TABLE, "area": AREA_TABLE}[kind]
    return pd.DataFrame(table_rows(fields, facts[(kind, sid)])).set_index("Factor")

## The network summary (the app opens with this)

In [3]:
show("uae", "uae")

### uae — **SUMMARY** (ai)

**6 PROTECT, 14 HOLD and 3 SHRINK across 23 scored lounges, each judged against its own emirate's salon sites, with 34 areas to GROW.**

- **shrink**: Investigate al-jada, city-walk, westyas: their sites are weak for their emirates on rent or demand (strongest SHRINK composite 0.35).  
  <small>SHRINK: 3; SHRINK lounges: al-jada, city-walk, westyas; Strongest SHRINK composite: 0.348</small>
- **grow**: 34 areas would each add 6,000+ net new women at a site that out-serves half its emirate's salon sites and costs no more than their median rent (or has only the emirate median rent, then low confidence), 796,024 women 15+ between them.  
  <small>GROW: 34; Women in GROW areas: 796024; Closest GROW area to a lounge: 1.3018; Furthest GROW area from a lounge: 54.9537</small>
- **protect**: Defend al-ain, al-dhafra, al-falah, al-taif-mall, mirdif-35, nad-al-sheba: strong sites for their emirates (weakest PROTECT composite 0.66, al-falah, on a neutral fallback rent).  
  <small>PROTECT: 6; PROTECT lounges: al-ain, al-dhafra, al-falah, al-taif-mall, mirdif-35, nad-al-sheba; Weakest PROTECT composite: 0.6645</small>
- **confidence**: 14 of 23 lounges sit in the HOLD band and 11 calls are low confidence (near a line, sensitive to the assumptions or weights, or on an emirate-median rent).  
  <small>HOLD: 14; Low-confidence calls: 11; Lounges scored: 23</small>
- **data_gaps**: No revenue, lease or footfall data: these are location and market calls, built on 3,833 salons found by search.  
  <small>Revenue data: False; Lease data: False; Salons searched: 3833</small>

_Lounge calls rank each site within its emirate's frozen market of competitor salon sites, on demand and rent; growth areas are what-if new lounges tested on net new women, demand and rent._

## Lounges, PROTECT first

In [4]:
order = {"PROTECT": 0, "HOLD": 1, "SHRINK": 2, "NOT SCORED": 3}
for k, sid, a, f in sorted((s for s in subjects if s[0] == "lounge"), key=lambda s: (order[s[2]], s[1])):
    show(k, sid)

### al-ain — **PROTECT** (ai)

**PROTECT: al-ain's site would out-serve 95% of Abu Dhabi salon sites on demand; its rent is only the Abu Dhabi median, scored neutral, so the call is low confidence (composite 0.72).**

- **demand**: A typical lounge at this site would serve about 10,173 women, more than it would at 95% of Abu Dhabi salon sites; 93rd percentile UAE-wide.  
  <small>Site demand percentile: 0.9474; Demand percentile UAE-wide: 0.9323; Women a typical lounge would serve here: 10173.4578; Demand score: 0.9474</small>
- **cost**: Its site rent of AED 1,168/m² a year is the Abu Dhabi median standing in for a cell with no rent of its own, not the site's, so rent scores neutral (0.5); for context it is dearer than 22% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent at the site: 1168.0; Cost score: 0.5; Rent percentile UAE-wide: 0.2155</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### al-dhafra — **PROTECT** (ai)

**PROTECT: al-dhafra's site would out-serve 100% of Abu Dhabi salon sites on demand, with a rent about level with the median salon site in its emirate (composite 0.76).**

- **demand**: A typical lounge at this site would serve about 22,852 women, more than it would at 100% of Abu Dhabi salon sites; 100th percentile UAE-wide.  
  <small>Site demand percentile: 1.0; Demand percentile UAE-wide: 0.9955; Women a typical lounge would serve here: 22851.8054; Demand score: 1.0</small>
- **cost**: Shop rent of AED 1,339/m² a year gives a rent ratio of 1.04, about level with the median salon site in its emirate; dearer than 32% of UAE salon sites.  
  <small>Rent ratio: 1.0411; Rent percentile UAE-wide: 0.3188; Shop rent at the site: 1339.0; Cost score: 0.5219</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### al-falah — **PROTECT** (ai)

**PROTECT: al-falah's site would out-serve 83% of Abu Dhabi salon sites on demand; its rent is only the Abu Dhabi median, scored neutral, so the call is low confidence (composite 0.66).**

- **demand**: A typical lounge at this site would serve about 7,866 women, more than it would at 83% of Abu Dhabi salon sites; 85th percentile UAE-wide.  
  <small>Site demand percentile: 0.8289; Demand percentile UAE-wide: 0.8522; Women a typical lounge would serve here: 7865.5012; Demand score: 0.8289</small>
- **cost**: Its site rent of AED 1,168/m² a year is the Abu Dhabi median standing in for a cell with no rent of its own, not the site's, so rent scores neutral (0.5); for context it is dearer than 22% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent at the site: 1168.0; Cost score: 0.5; Rent percentile UAE-wide: 0.2155</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### al-taif-mall — **PROTECT** (ai)

**PROTECT: al-taif-mall's site would out-serve 90% of UAE salon sites on demand, with a rent cheaper than the median salon site in its emirate (composite 0.95).**

- **cost**: Shop rent of AED 481/m² a year gives a rent ratio of 1.55, cheaper than the median salon site in its emirate; among the cheapest UAE salon sites.  
  <small>Rent ratio: 1.5506; Rent percentile UAE-wide: 0.0045; Shop rent at the site: 480.6; Cost score: 1.0</small>
- **demand**: A typical lounge at this site would serve about 8,980 women, more than it would at 90% of UAE salon sites (under 50 reference sites in its emirate, so ranked UAE-wide); 90th percentile UAE-wide.  
  <small>Site demand percentile: 0.9003; Demand percentile UAE-wide: 0.9003; Women a typical lounge would serve here: 8980.4181; Demand score: 0.9003</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### mirdif-35 — **PROTECT** (ai)

**PROTECT: mirdif-35's site would out-serve 98% of Dubai salon sites on demand, with a rent cheaper than the median salon site in its emirate (composite 0.79).**

- **demand**: A typical lounge at this site would serve about 5,388 women, more than it would at 98% of Dubai salon sites; 75th percentile UAE-wide.  
  <small>Site demand percentile: 0.9845; Demand percentile UAE-wide: 0.7498; Women a typical lounge would serve here: 5388.3446; Demand score: 0.9845</small>
- **cost**: Shop rent of AED 1,957/m² a year gives a rent ratio of 1.10, cheaper than the median salon site in its emirate; dearer than 58% of UAE salon sites.  
  <small>Rent ratio: 1.1004; Rent percentile UAE-wide: 0.5815; Shop rent at the site: 1957.0; Cost score: 0.595</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

...

### zawaya-walk — **HOLD** (ai)

**HOLD: zawaya-walk's site would out-serve 58% of Sharjah salon sites on demand, with a rent dearer than the median salon site in its emirate (composite 0.41).**

- **cost**: Shop rent of AED 1,389/m² a year gives a rent ratio of 0.81, dearer than the median salon site in its emirate; dearer than 36% of UAE salon sites.  
  <small>Rent ratio: 0.8071; Rent percentile UAE-wide: 0.3562; Shop rent at the site: 1389.1; Cost score: 0.2335</small>
- **demand**: A typical lounge at this site would serve about 7,569 women, more than it would at 58% of Sharjah salon sites; 83rd percentile UAE-wide.  
  <small>Site demand percentile: 0.5804; Demand percentile UAE-wide: 0.8344; Women a typical lounge would serve here: 7568.9551; Demand score: 0.5804</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### al-jada — **SHRINK** (ai)

**SHRINK: al-jada's site would out-serve 66% of Sharjah salon sites on demand, with a rent dearer than the median salon site in its emirate (composite 0.34).**

- **cost**: Shop rent of AED 1,771/m² a year gives a rent ratio of 0.63, dearer than the median salon site in its emirate; dearer than 54% of UAE salon sites.  
  <small>Rent ratio: 0.6331; Rent percentile UAE-wide: 0.5423; Shop rent at the site: 1770.8; Cost score: 0.0191</small>
- **demand**: A typical lounge at this site would serve about 7,912 women, more than it would at 66% of Sharjah salon sites; 85th percentile UAE-wide.  
  <small>Site demand percentile: 0.6573; Demand percentile UAE-wide: 0.8531; Women a typical lounge would serve here: 7912.2548; Demand score: 0.6573</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### city-walk — **SHRINK** (ai)

**SHRINK: city-walk's site would out-serve 30% of Dubai salon sites on demand, with a rent dearer than the median salon site in its emirate (composite 0.15).**

- **cost**: Shop rent of AED 4,561/m² a year gives a rent ratio of 0.47, dearer than the median salon site in its emirate; dearer than 98% of UAE salon sites.  
  <small>Rent ratio: 0.4722; Rent percentile UAE-wide: 0.9849; Shop rent at the site: 4561.0; Cost score: 0.0</small>
- **demand**: A typical lounge at this site would serve about 1,905 women, more than it would at 30% of Dubai salon sites; 26th percentile UAE-wide.  
  <small>Site demand percentile: 0.3024; Demand percentile UAE-wide: 0.2591; Women a typical lounge would serve here: 1905.0265; Demand score: 0.3024</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### westyas — **SHRINK** (ai)

**SHRINK: westyas's site would out-serve 45% of Abu Dhabi salon sites on demand, with a rent dearer than the median salon site in its emirate (composite 0.348).**

- **cost**: Shop rent of AED 1,707/m² a year gives a rent ratio of 0.82, dearer than the median salon site in its emirate; dearer than 52% of UAE salon sites.  
  <small>Rent ratio: 0.8166; Rent percentile UAE-wide: 0.5165; Shop rent at the site: 1707.0; Cost score: 0.2453</small>
- **demand**: A typical lounge at this site would serve about 3,259 women, more than it would at 45% of Abu Dhabi salon sites; 57th percentile UAE-wide.  
  <small>Site demand percentile: 0.4507; Demand percentile UAE-wide: 0.5672; Women a typical lounge would serve here: 3259.3995; Demand score: 0.4507</small>

_Both scored signals rank this lounge's site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve, and the rent ratio is their median shop rent over this site's (above 1 is cheaper). The UAE-wide percentiles are context; served women, capture, rating and the shared catchment are shown, not scored._

### zayed-international-airport — **NOT SCORED** (template)

**Not scored: this lounge serves travellers, not the women living around it, so the catchment model doesn't apply.**

- **demand**: Site demand percentile: n/a (missing), shown but not scored.  
  <small>Site demand percentile: None; Demand percentile UAE-wide: None; Demand ranked among: Abu Dhabi; Women a typical lounge would serve here: None; Women served (Huff): None</small>
- **capture**: Capture: 4% of reviews, shown but not scored.  
  <small>Capture: 0.0392; Premium substitutes: 11; Premium salons in catchment: 51; Thin premium market: False</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Both rank the site within its emirate's frozen market of competitor salon sites: demand is the share of them a typical lounge here would out-serve (Huff); rent is their median shop rent over this site's. The lounge's own served women, as performance vs its site, capture, rating and the shared catchment are shown for context, not scored._

The factor table the app shows next to a lounge's pyramid, for one example:

In [5]:
table("lounge", "al-barsha")

,Value,Unit,Threshold,What it means
Factor,,,,
Site demand percentile,42%,% of salon sites,"Demand score: 0% of salon sites → 0, 100% of salon sites → 1; weight 1",Share of the reference salon sites in its emirate (UAE-wide under 50) that a typical lounge would out-serve at this ...
Demand ranked among,Dubai,,—,"Whose salon sites the demand percentile ranks against: the site's emirate, or UAE when the emirate has under 50 refe..."
Demand percentile UAE-wide,33%,% of UAE salon sites,—,"The same, against all 1,123 UAE reference salon sites. Context, not scored."
Women a typical lounge would serve here,"2,192",women,—,"What-if: the women a typical lounge (median reviews and rating) would serve at this site, with this lounge closed an..."
Women served (Huff),"2,180",women,—,Addressable women this lounge serves: its Huff share of every cell it reaches within 20 min. It rests on the lounge'...
Performance vs site,0.99,x its site,—,Women this lounge serves ÷ women a typical lounge would serve at its site. Read as 'The model estimates this lounge ...
Absorbed by siblings,1%,% of served,—,Share of its served women other lounges take over if it closes.
Women 15+ in catchment,"272,371",women,—,"Women aged 15+ living in the ~2 km cells within a 15-min drive (typical midday traffic). WorldPop adults, rebalanced..."
Addressable women 15+,"272,371",women,—,"Catchment women times an affluence weight (observed median rent vs the women-weighted median, to the affluence elast..."


## Growth areas: GROW, then WATCH by women

In [6]:
areas = sorted((s for s in subjects if s[0] == "area" and s[2] != "SKIP"),
               key=lambda s: (s[2] != "GROW", -s[3]["women"]))
for k, sid, a, f in areas:
    show(k, sid)

### sharjah-sharjah — **GROW** (ai)

**GROW Sharjah: a new lounge would add about 12,800 net new women, out-serve 90% of Sharjah salon sites and pay a rent no dearer than their median.**

- **size**: A new lounge would serve 13,176 women, take 368 from our lounges and add 12,808 net, 6,808 above the 6,000 line.  
  <small>Net new women: 12808.3802; New lounge would serve: 13176.3868; Taken from our lounges: 368.0067; Size gap: 6808</small>
- **demand**: That is more than a typical lounge would serve at 90% of Sharjah salon sites, 98th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.9021; Demand gap: 0.4021; Demand percentile UAE-wide: 0.9777; New lounge would serve: 13176.3868</small>
- **cost**: Shop rent of AED 1,121/m² a year gives a rent ratio of 1.00 against the median Sharjah salon site, exactly on the 1 line; dearer than 20% of UAE salon sites.  
  <small>Rent ratio: 1.0; Rent gap: 0.0; Rent percentile UAE-wide: 0.195; Shop rent in the area: 1121.1</small>
- **reach**: Our al-jada lounge is only 2.5 km away in a straight line, yet a new site would take just 368 women from our lounges.  
  <small>Distance to nearest lounge: 2.4824; Nearest lounge: al-jada</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median; the UAE-wide percentiles are context._

### al-dhaid-sharjah — **GROW** (ai)

**GROW Al Dhaid: a new lounge would add about 57,900 net new women and out-serve 100% of Sharjah salon sites; its rent is only the Sharjah median, so the rent test is neutral and the call low confidence.**

- **size**: A new lounge would serve 57,932 women and take almost none from our lounges, so 57,932 are net new, 51,932 above the 6,000 line.  
  <small>Net new women: 57931.7203; New lounge would serve: 57931.7203; Taken from our lounges: 0.0; Size gap: 51932</small>
- **demand**: That is more than a typical lounge would serve at 100% of Sharjah salon sites, 100th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 1.0; Demand gap: 0.5; Demand percentile UAE-wide: 1.0; New lounge would serve: 57931.7203</small>
- **cost**: Shop rent of AED 1,121/m² a year is the Sharjah median standing in for cells with no rent of their own, not an observation of this place, so the rent test is neutral; for context it is dearer than 20% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent in the area: 1121.1; Rent percentile UAE-wide: 0.195</small>
- **reach**: The nearest lounge, al-jada, is 43.8 km away: Al Dhaid is white space for the network.  
  <small>Distance to nearest lounge: 43.7824; Nearest lounge: al-jada</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median, unless that rent is only the emirate median standing in for cells without one: then the rent test is neutral and the call low confidence._

### kalba-sharjah — **GROW** (ai)

**GROW Kalba: a new lounge would add about 12,500 net new women and out-serve 90% of Sharjah salon sites; its rent is only the Sharjah median, so the rent test is neutral and the call low confidence.**

- **size**: A new lounge would serve 13,329 women, take 871 from our lounges and add 12,459 net, 6,459 above the 6,000 line.  
  <small>Net new women: 12458.6472; New lounge would serve: 13329.2141; Taken from our lounges: 870.5669; Size gap: 6459</small>
- **demand**: That is more than a typical lounge would serve at 90% of Sharjah salon sites, 98th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.9021; Demand gap: 0.4021; Demand percentile UAE-wide: 0.9786; New lounge would serve: 13329.2141</small>
- **cost**: Shop rent of AED 1,121/m² a year is the Sharjah median standing in for cells with no rent of their own, not an observation of this place, so the rent test is neutral; for context it is dearer than 20% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent in the area: 1121.1; Rent percentile UAE-wide: 0.195</small>
- **reach**: The nearest lounge, al-taif-mall, is 9.0 km away in a straight line.  
  <small>Distance to nearest lounge: 9.0015; Nearest lounge: al-taif-mall</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median, unless that rent is only the emirate median standing in for cells without one: then the rent test is neutral and the call low confidence._

### al-madam-sharjah — **GROW** (ai)

**GROW Al Madam: a new lounge would add about 19,700 net new women and out-serve 94% of Sharjah salon sites; its rent is only the Sharjah median, so the rent test is neutral and the call low confidence.**

- **size**: A new lounge would serve 19,728 women and take almost none from our lounges, so 19,728 are net new, 13,728 above the 6,000 line.  
  <small>Net new women: 19727.7719; New lounge would serve: 19727.7719; Taken from our lounges: 0.0; Size gap: 13728</small>
- **demand**: That is more than a typical lounge would serve at 94% of Sharjah salon sites, 99th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.9441; Demand gap: 0.4441; Demand percentile UAE-wide: 0.9929; New lounge would serve: 19727.7719</small>
- **cost**: Shop rent of AED 1,121/m² a year is the Sharjah median standing in for cells with no rent of their own, not an observation of this place, so the rent test is neutral; for context it is dearer than 20% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent in the area: 1121.1; Rent percentile UAE-wide: 0.195</small>
- **reach**: The nearest lounge, mirdif-35, is 46.7 km away: Al Madam is white space for the network.  
  <small>Distance to nearest lounge: 46.7161; Nearest lounge: mirdif-35</small>
- **data_gap**: Our salon search covered only 37% of the women here (a confident call needs 50%), so part of the competition is imputed.  
  <small>Competitor data coverage: 0.3662; Coverage needed: 0.5; Coverage gap: -0.1338</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median, unless that rent is only the emirate median standing in for cells without one: then the rent test is neutral and the call low confidence._

### khor-fakkan-sharjah — **GROW** (ai)

**GROW Khor Fakkan: a new lounge would add about 23,400 net new women and out-serve 97% of Sharjah salon sites; its rent is only the Sharjah median, so the rent test is neutral and the call low confidence.**

- **size**: A new lounge would serve 23,433 women, take 8 from our lounges and add 23,424 net, 17,424 above the 6,000 line.  
  <small>Net new women: 23424.3675; New lounge would serve: 23432.6178; Taken from our lounges: 8.2504; Size gap: 17424</small>
- **demand**: That is more than a typical lounge would serve at 97% of Sharjah salon sites, 100th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.972; Demand gap: 0.472; Demand percentile UAE-wide: 0.9964; New lounge would serve: 23432.6178</small>
- **cost**: Shop rent of AED 1,121/m² a year is the Sharjah median standing in for cells with no rent of their own, not an observation of this place, so the rent test is neutral; for context it is dearer than 20% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent in the area: 1121.1; Rent percentile UAE-wide: 0.195</small>
- **reach**: The nearest lounge, al-taif-mall, is 26.5 km away in a straight line.  
  <small>Distance to nearest lounge: 26.5027; Nearest lounge: al-taif-mall</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median, unless that rent is only the emirate median standing in for cells without one: then the rent test is neutral and the call low confidence._

...

### madinat-al-riyad-abu-dhabi — **WATCH** (ai)

**WATCH Madinat Al Riyad: a new lounge would add 5,116 net new women, 884 short of the 6,000 line.**

- **size**: A new lounge would serve 6,876 women, take 1,760 from our lounges and add 5,116 net, 884 short of the 6,000 line.  
  <small>Net new women: 5116.0955; New lounge would serve: 6876.2738; Taken from our lounges: 1760.1783; Size gap: -884</small>
- **demand**: That is more than a typical lounge would serve at 78% of Abu Dhabi salon sites, 81st percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.7829; Demand gap: 0.2829; Demand percentile UAE-wide: 0.8121; New lounge would serve: 6876.2738</small>
- **cost**: Shop rent of AED 2,822/m² a year gives a rent ratio of 0.49 against the median Abu Dhabi salon site, 0.51 short of the 1 line; dearer than 90% of UAE salon sites.  
  <small>Rent ratio: 0.494; Rent gap: -0.506; Rent percentile UAE-wide: 0.9029; Shop rent in the area: 2822.0</small>
- **reach**: The nearest lounge, al-falah, is 10.3 km away in a straight line.  
  <small>Distance to nearest lounge: 10.2547; Nearest lounge: al-falah</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median; the UAE-wide percentiles are context._

### al-mizhar-2-dubai — **WATCH** (ai)

**WATCH Al Mizhar 2: a new lounge would add 4,411 net new women, 1,589 short of the 6,000 line.**

- **size**: A new lounge would serve 4,551 women, take 140 from our lounges and add 4,411 net, 1,589 short of the 6,000 line.  
  <small>Net new women: 4411.17; New lounge would serve: 4551.0769; Taken from our lounges: 139.9069; Size gap: -1589</small>
- **demand**: That is more than a typical lounge would serve at 96% of Dubai salon sites, 70th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.9622; Demand gap: 0.4622; Demand percentile UAE-wide: 0.7017; New lounge would serve: 4551.0769</small>
- **cost**: Shop rent of AED 3,502/m² a year gives a rent ratio of 0.61 against the median Dubai salon site, 0.39 short of the 1 line; dearer than 95% of UAE salon sites.  
  <small>Rent ratio: 0.6149; Rent gap: -0.3851; Rent percentile UAE-wide: 0.9475; Shop rent in the area: 3502.0</small>
- **reach**: The nearest lounge, mirdif-35, is 3.6 km away in a straight line.  
  <small>Distance to nearest lounge: 3.6387; Nearest lounge: mirdif-35</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median; the UAE-wide percentiles are context._

### al-kheeran-dubai — **WATCH** (ai)

**WATCH Al Kheeran: a new lounge would add 3,558 net new women, 2,442 short of the 6,000 line.**

- **size**: A new lounge would serve 3,631 women, take 73 from our lounges and add 3,558 net, 2,442 short of the 6,000 line.  
  <small>Net new women: 3557.5706; New lounge would serve: 3630.8237; Taken from our lounges: 73.2532; Size gap: -2442</small>
- **demand**: That is more than a typical lounge would serve at 87% of Dubai salon sites, 61st percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.8711; Demand gap: 0.3711; Demand percentile UAE-wide: 0.6144; New lounge would serve: 3630.8237</small>
- **cost**: Shop rent of AED 2,746/m² a year gives a rent ratio of 0.78 against the median Dubai salon site, 0.22 short of the 1 line; dearer than 89% of UAE salon sites.  
  <small>Rent ratio: 0.7842; Rent gap: -0.2158; Rent percentile UAE-wide: 0.8949; Shop rent in the area: 2746.0</small>
- **reach**: The nearest lounge, nad-al-sheba, is 6.6 km away in a straight line.  
  <small>Distance to nearest lounge: 6.6295; Nearest lounge: nad-al-sheba</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median; the UAE-wide percentiles are context._

### al-marmoom-dubai — **WATCH** (ai)

**WATCH Al Marmoom: a new lounge would add 4,493 net new women, 1,507 short of the 6,000 line.**

- **size**: A new lounge would serve 4,493 women and take almost none from our lounges, so 4,493 are net new, 1,507 short of the 6,000 line.  
  <small>Net new women: 4492.5799; New lounge would serve: 4492.7294; Taken from our lounges: 0.1495; Size gap: -1507</small>
- **demand**: That is more than a typical lounge would serve at 96% of Dubai salon sites, 70th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.9588; Demand gap: 0.4588; Demand percentile UAE-wide: 0.6972; New lounge would serve: 4492.7294</small>
- **cost**: Shop rent of AED 1,985/m² a year is the Dubai median standing in for cells with no rent of their own, not an observation of this place, so the rent test is neutral; for context it is dearer than 60% of UAE salon sites.  
  <small>Rent is the emirate median: True; Shop rent in the area: 1985.0; Rent percentile UAE-wide: 0.5975</small>
- **reach**: The nearest lounge, nad-al-sheba, is 22.7 km away in a straight line.  
  <small>Distance to nearest lounge: 22.7169; Nearest lounge: nad-al-sheba</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median, unless that rent is only the emirate median standing in for cells without one: then the rent test is neutral and the call low confidence._

### al-muwaihat-ajman — **WATCH** (ai)

**WATCH Al Muwaihat: a new lounge would add about 11,700 net new women, but its rent is dearer than the median Ajman salon site.**

- **size**: A new lounge would serve 11,954 women, take 238 from our lounges and add 11,716 net, 5,716 above the 6,000 line.  
  <small>Net new women: 11715.7382; New lounge would serve: 11953.9612; Taken from our lounges: 238.223; Size gap: 5716</small>
- **demand**: That is more than a typical lounge would serve at 96% of UAE salon sites (Ajman has under 50, so ranked UAE-wide), 96th percentile UAE-wide; GROW needs 50%.  
  <small>Site demand percentile: 0.9626; Demand gap: 0.4626; Demand percentile UAE-wide: 0.9626; New lounge would serve: 11953.9612</small>
- **cost**: Shop rent of AED 821/m² a year gives a rent ratio of 0.92 against the median Ajman salon site, 0.08 short of the 1 line; dearer than 8% of UAE salon sites.  
  <small>Rent ratio: 0.9174; Rent gap: -0.0826; Rent percentile UAE-wide: 0.0766; Shop rent in the area: 820.8</small>
- **reach**: The nearest lounge, al-jada, is 7.3 km away in a straight line.  
  <small>Distance to nearest lounge: 7.2531; Nearest lounge: al-jada</small>

_Each area is a what-if: open a typical lounge here and count the women it adds to the network, net of what it takes from our own lounges. GROW also needs the site to out-serve at least half of its emirate's salon sites and a rent no dearer than their median; the UAE-wide percentiles are context._

In [7]:
table("area", areas[0][1])

,Value,Unit,Threshold,What it means
Factor,,,,
Women 15+,"281,946",women,"SKIP under 5,000",Women aged 15+ living in the area's cells (raw).
Addressable women 15+,"281,946",women,—,"Catchment women times an affluence weight (observed median rent vs the women-weighted median, to the affluence elast..."
Affluence rent,n/a (missing),AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,0%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells,89,count,—,Populated ~2 km cells in the area (every unreached cell with this OSM place name).
Worker housing,12%,% of adults,GROW needs under 50%,Share of the area's adults living in worker housing (OSM industrial land use).
Premium salons,107,count,—,Premium salons found in the area's searched cells.
Premium saturation,116,reviews per 1k women,"Unsaturated under 150 (context, not a test)","Premium salons' Google reviews (scaled up for missed salons) per 1,000 women in the searched cells: the same measure..."
Shop rent in the area,"1,121",AED/m²/yr,—,"Women-weighted median shop rent over the area's cells (DLD in Dubai, ADREC in Abu Dhabi, listings in the northern em..."


## Play with it

Change a number and see the fallback: the cache key is a hash of the fact sheet, so any change
means the template is served instead of the cached AI text (and `verify` would reject the old
text anyway).

In [8]:
k, sid, a, f = next(s for s in subjects if s[1] == "al-barsha")
changed = {**f, "catchment_women": f["catchment_women"] + 1}
print(explain(k, sid, a, changed, cache=cache).source, "<- template, because one number changed")

template <- template, because one number changed
